# Taller — Tu modelo de Machine Learning como herramienta de un agente de IA

**Unidad 2 — Estrategias de Transformación Digital**

## Objetivo del taller

Ya entrenaste un modelo de Machine Learning simple sobre una base de datos de tu elección (el ejercicio anterior). En este taller vas a:

1. Traer ese modelo (código + pesos entrenados) a este notebook.
2. Convertirlo en una **herramienta** que un agente de IA pueda invocar (tool calling, visto en la Unidad 2.3).
3. Construir un agente real (con modelo de lenguaje Gemini) que use tu modelo como una de sus herramientas.
4. Conversar con el agente pidiéndole recomendaciones, y **exigirle que justifique por qué** llegó a esa recomendación — no basta con que repita el número que arrojó el modelo.
5. Documentar, en tus propias palabras, la decisión de negocio que tomarías a partir de cada interacción.

```{admonition} Idea central del taller
:class: tip
Un modelo de ML por sí solo solo predice. Un agente que usa ese modelo como herramienta puede además **razonar sobre el resultado, explicarlo en lenguaje de negocio y recomendar una acción** — esa es la diferencia entre "tener un modelo" y "tener un sistema de decisión" dentro de una organización.
```

## Cómo está organizado este notebook

Cada sección tiene celdas marcadas con **`# TODO`** o **`# EJEMPLO A REEMPLAZAR`**. Las celdas de ejemplo ya corren de principio a fin con un caso de churn (cancelación de clientes) ficticio, para que veas el flujo completo funcionando antes de poner tu propio modelo. Tu trabajo es reemplazar esas celdas por tu propio pipeline, sin cambiar la estructura general del notebook.


## Paso 0 — Dependencias

Si vas a correr esto en Google Colab, instala el SDK de Gemini (las demás librerías ya vienen preinstaladas).

In [ ]:
# Si estás en Colab, descomenta la siguiente línea:
# !pip install -q google-genai

import pandas as pd
import numpy as np
import joblib
import json


## Paso 1 — Pega aquí tu pipeline de Machine Learning

Trae el código que ya hiciste: cargar tus datos, preprocesarlos (si aplica) y entrenar tu modelo. No tiene que ser sofisticado — el taller anterior pedía justamente un ejercicio simple.

**Lo único que importa para los pasos siguientes es que, al terminar esta sección, tengas una variable con tu modelo ya entrenado** (en el ejemplo se llama `modelo`). Si tu variable se llama distinto, ajústala o renómbrala al final de esta celda.

In [ ]:
# ============================================================
# EJEMPLO A REEMPLAZAR — bórralo y pega aquí tu propio código
# ============================================================
# Dataset de ejemplo: predicción de cancelación de clientes (churn)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
n = 500
datos_ejemplo = pd.DataFrame({
    "antiguedad_meses": np.random.randint(1, 72, n),
    "quejas_registradas": np.random.poisson(1.2, n),
    "gasto_mensual": np.round(np.random.normal(50, 15, n), 2),
})
# Variable objetivo simulada con una relación simple + ruido
probabilidad_cancela = (
    0.4
    - 0.004 * datos_ejemplo["antiguedad_meses"]
    + 0.15 * datos_ejemplo["quejas_registradas"]
    + np.random.normal(0, 0.1, n)
)
datos_ejemplo["cancela"] = (probabilidad_cancela > 0.3).astype(int)

X = datos_ejemplo[["antiguedad_meses", "quejas_registradas", "gasto_mensual"]]
y = datos_ejemplo["cancela"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
print(f"Exactitud en datos de prueba: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))


## Paso 2 — Guarda los pesos de tu modelo

Guardamos el modelo entrenado en disco con `joblib`, para que la función-herramienta del Paso 3 pueda cargarlo sin tener que reentrenarlo cada vez que el agente la invoque (en producción real nunca querrías reentrenar un modelo cada vez que alguien lo consulta).

In [ ]:
NOMBRE_ARCHIVO_MODELO = "mi_modelo.pkl"
joblib.dump(modelo, NOMBRE_ARCHIVO_MODELO)
print(f"Modelo guardado en: {NOMBRE_ARCHIVO_MODELO}")


## Paso 3 — Convierte tu modelo en una herramienta (tool)

Recuerda de la Unidad 2.3: una herramienta para un agente es simplemente una función de Python con tres requisitos:

1. **Parámetros con tipo explícito** (`float`, `int`, `str`, `bool`) — el SDK de Gemini usa esos tipos para generar automáticamente el esquema que el modelo de lenguaje necesita para saber qué datos pedir.
2. **Un docstring claro** que explique qué hace la función y qué representa cada parámetro — el modelo de lenguaje lee literalmente ese texto para decidir cuándo y cómo usar la herramienta, así que entre más claro y específico en términos de negocio, mejor.
3. **Un resultado serializable** (un diccionario con tipos simples) — nunca devuelvas directamente un objeto de scikit-learn o un array de numpy.

**Ajusta la firma de la función (los parámetros) para que coincida exactamente con las variables (features) que usa tu propio modelo.**

In [ ]:
# ============================================================
# EJEMPLO A REEMPLAZAR — ajusta los parámetros a tus propias
# variables (features) y la lógica de negocio a tu caso
# ============================================================

def predecir_cancelacion_cliente(
    antiguedad_meses: float,
    quejas_registradas: float,
    gasto_mensual: float,
) -> dict:
    """Predice si un cliente va a cancelar su servicio (churn), a partir de su
    antigüedad, número de quejas registradas y gasto mensual promedio.

    Usa esta herramienta cuando el usuario pregunte si un cliente específico
    tiene riesgo de cancelar, o pida evaluar el riesgo de cancelación de un
    cliente con características dadas.

    Args:
        antiguedad_meses: Número de meses que el cliente lleva activo.
        quejas_registradas: Número de quejas formales que el cliente ha
            registrado en el histórico.
        gasto_mensual: Gasto mensual promedio del cliente, en la moneda local.

    Returns:
        Diccionario con la predicción ("cancela" o "no_cancela"), la
        probabilidad estimada de cancelación, y las variables usadas.
    """
    modelo_cargado = joblib.load(NOMBRE_ARCHIVO_MODELO)

    entrada = pd.DataFrame([{
        "antiguedad_meses": antiguedad_meses,
        "quejas_registradas": quejas_registradas,
        "gasto_mensual": gasto_mensual,
    }])

    prediccion = modelo_cargado.predict(entrada)[0]
    probabilidad = modelo_cargado.predict_proba(entrada)[0][1]  # prob. de clase "cancela"

    return {
        "prediccion": "cancela" if prediccion == 1 else "no_cancela",
        "probabilidad_cancela": round(float(probabilidad), 3),
        "antiguedad_meses": antiguedad_meses,
        "quejas_registradas": quejas_registradas,
        "gasto_mensual": gasto_mensual,
    }


### Prueba tu función de forma aislada

Antes de dársela a un agente, siempre valida que la función funciona sola — si falla aquí, va a fallar igual (y de forma más difícil de depurar) cuando el agente intente usarla.

In [ ]:
# EJEMPLO A REEMPLAZAR — usa valores razonables para tus propias variables
resultado_prueba = predecir_cancelacion_cliente(
    antiguedad_meses=3,
    quejas_registradas=4,
    gasto_mensual=35.0,
)
print(json.dumps(resultado_prueba, indent=2, ensure_ascii=False))


## Paso 4 — Configura el agente (Gemini)

Igual que en el notebook de la Unidad 2.3: en Colab, guarda tu API key en "Secrets" (el ícono de llave) con el nombre `GEMINI_API_KEY`. Si corres esto localmente, te la pedirá por teclado.

In [ ]:
from google import genai
from google.genai import types
import os

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except ImportError:
    import getpass
    api_key = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Pega tu API key de Gemini: ")

client = genai.Client(api_key=api_key)
MODELO_GEMINI = "gemini-3.8-flash"


## Paso 5 — Declara tu función como herramienta del agente

Usamos **function calling automático**: le pasamos la función de Python directamente en `tools`, y el SDK de Gemini genera el esquema por nosotros a partir de los tipos y el docstring.

La parte clave de esta sección es el `system_instruction`: ahí le exigimos explícitamente al agente que **no se limite a repetir el resultado crudo de la herramienta**, sino que explique su razonamiento y lo traduzca a una recomendación de negocio — esto es lo que separa un agente útil de uno que simplemente reenvía un número.

In [ ]:
INSTRUCCION_SISTEMA = """
"""

config_agente = types.GenerateContentConfig(
    tools=[predecir_cancelacion_cliente],
    system_instruction=INSTRUCCION_SISTEMA,
)


## Paso 6 — Conversa con tu agente

Usamos un chat (que mantiene el historial de la conversación) en lugar de llamadas sueltas, para poder hacer preguntas de seguimiento como "¿por qué dices eso?" en el Paso 7.

In [ ]:
chat = client.chats.create(model=MODELO_GEMINI, config=config_agente)

respuesta_1 = chat.send_message(
    "Tengo un cliente que lleva 3 meses con nosotros, ha puesto 4 quejas "
    "y gasta en promedio 35 dólares al mes. ¿Qué tan probable es que cancele, "
    "y qué deberíamos hacer?"
)
print(respuesta_1.text)


## Paso 7 — Cuestiona la decisión del agente

Un buen ejercicio de pensamiento crítico (y una práctica real de negocio) es no aceptar la primera recomendación sin más: pregúntale al agente por qué llegó a esa conclusión, o qué cambiaría su recomendación. Esto también sirve para detectar si el agente está **alucinando** una justificación que no se sostiene en los datos reales que usó la herramienta (riesgo visto en la Unidad 2.3).

In [ ]:
respuesta_2 = chat.send_message(
    "¿Por qué llegaste exactamente a esa recomendación? "
    "¿Qué pasaría con tu recomendación si el cliente no tuviera ninguna queja registrada?"
)
print(respuesta_2.text)


Compara: ¿la nueva respuesta del agente vuelve a llamar la herramienta con el escenario hipotético (0 quejas) para fundamentar su respuesta, o simplemente especula en texto sin volver a consultar el modelo? Si tu agente está bien instruido, debería preferir llamar la herramienta de nuevo con el nuevo escenario antes de especular — anótalo en tu análisis del Paso 8.

## Paso 8 — Documenta tus decisiones de negocio

Repite el Paso 6 y el Paso 7 con **al menos tres clientes/casos distintos** usando tu propio modelo (no el de ejemplo). Para cada uno, completa la tabla de abajo en una celda de markdown, con tus propias palabras — este es el entregable central del taller: no basta con que el agente funcione, tienes que mostrar que entendiste y evaluaste críticamente su razonamiento.

| Caso | Resultado del modelo | Explicación del agente (resumida) | ¿La explicación es consistente con los datos? | Decisión de negocio que tomarías |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

```{admonition} Criterio de evaluación
:class: important
No se evalúa solo que el notebook corra. Se evalúa que: (1) tu modelo quedó correctamente envuelto como herramienta, (2) el `system_instruction` realmente obliga al agente a justificar y no solo repetir el número, (3) cuestionaste al menos una respuesta del agente (Paso 7), y (4) tu tabla del Paso 8 muestra pensamiento crítico real, incluyendo si en algún caso detectaste que la "justificación" del agente no se sostenía en los datos.
```

In [ ]:
# Completa aquí la tabla de tus tres casos (o dupĺicala en una celda markdown si prefieres)
# Caso 1
respuesta_caso_1 = chat.send_message("Describe aquí a tu primer cliente/caso real")
print(respuesta_caso_1.text)


In [ ]:
# Caso 2
respuesta_caso_2 = chat.send_message("Describe aquí a tu segundo cliente/caso real")
print(respuesta_caso_2.text)


In [ ]:
# Caso 3
respuesta_caso_3 = chat.send_message("Describe aquí a tu tercer cliente/caso real")
print(respuesta_caso_3.text)


## Entregable del taller

Entrega este notebook con:

1. Tu propio pipeline de ML en el Paso 1 (reemplazando el ejemplo de churn).
2. Tu función-herramienta en el Paso 3, con la firma y el docstring ajustados a tus variables.
3. El `system_instruction` del Paso 5 (puedes ajustar el texto, pero debe seguir exigiendo las tres partes: resultado, por qué, recomendación).
4. Al menos tres interacciones completas con el agente (Paso 8), incluyendo al menos un intento de cuestionar su respuesta (como en el Paso 7).
5. La tabla de decisiones de negocio del Paso 8, completa y en tus propias palabras.
